# Projeto Integrador VI (SI-PI6-2026-T012)
## CorretorIA: Sistema Inteligente de Precificação Imobiliária com Aprendizado de Máquina

* **Curso:** Bacharelado em Sistemas de Informação
* **Turma:** 012 - 2026
* **Tema:** Modelo de Precificação Hedônica e Apoio à Tomada de Decisão para Corretores de Imóveis
* **Foco da 1ª Entrega:** Validação do Modelo Piloto em São Paulo (SP) com Arquitetura Escalável Multi-Estado

---
### 📌 Contexto & Problema de Negócio
No mercado imobiliário, a precificação errática é a principal causa de imóveis que ficam meses estagnados sem proposta ou que geram prejuízo na liquidação rápida. 
Proprietários costumam superestimar o valor do bem devido a apego emocional, enquanto compradores buscam pechinchas agressivas. 

O **CorretorIA** resolve esse problema ao fornecer ao consultor imobiliário um embasamento técnico:
1. **Preço Justo Central de Mercado (R$)**
2. **Faixa Estratégica de Negociação** (Mínimo de liquidez acelerada vs. Teto negociável)
3. **Simulador em Tempo Real dos Honorários/Comissão do Corretor** (tabela CRECI de 6% ou customizada)
4. **Parecer Técnico com Explicabilidade** (peso da localização, metragem, idade do imóvel e vagas de garagem)


## 1. Importação das Bibliotecas
Importamos as ferramentas essenciais para processamento de dados, modelagem matemática, regressão e visualização gráfica.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Machine Learning & Métricas
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

print("Bibliotecas importadas com sucesso!")


## 2. Ingestão da Base de Dados de São Paulo
* **Origem dos Dados:** Base histórica com **7.500 registros** de transações da capital paulista cobrindo a série de **2013 a 2026**, calibrada nos padrões públicos de transações de ITBI (Prefeitura de SP e GeoSampa).
* **Decisão Metodológica (Foco em SP):** Como a demanda do orientador é de cobertura ampla, adotamos o mercado de São Paulo como base piloto pela alta liquidez e volume de dados limpos, deixando a modelagem estruturada para receber dados de outros estados (MG, SC, etc.) nas próximas etapas.


In [ ]:
dataset_path = os.path.join("data", "dataset_sp_imoveis.csv")

if not os.path.exists(dataset_path):
    # Se rodado de dentro de outra pasta
    dataset_path = os.path.abspath(os.path.join("..", "data", "dataset_sp_imoveis.csv"))

df = pd.read_csv(dataset_path)

print(f"Total de registros carregados: {len(df):,}")
print(f"Dimensões do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas\n")
df.head(5)


## 3. Análise Exploratória de Dados (EDA)
Inspecionamos a distribuição das variáveis: área útil, idade dos imóveis, cômodos, preço e valor do metro quadrado.


In [ ]:
# Resumo estatístico das variáveis numéricas
cols_interesse = ['area_m2', 'idade_imovel', 'quartos', 'banheiros', 'vagas', 'preco_m2', 'preco']
df[cols_interesse].describe().round(2)


### 3.1 Ranking de Bairros por Preço Médio do Metro Quadrado
Agrupamos os dados por bairro para entender a hierarquia de valorização em São Paulo.


In [ ]:
# Agrupamento por bairro
ranking_bairros = df.groupby('bairro').agg(
    preco_m2_medio=('preco_m2', 'mean'),
    preco_total_medio=('preco', 'mean'),
    total_imoveis=('id', 'count')
).reset_index().sort_values(by='preco_m2_medio', ascending=False)

print("Top 10 Bairros com maior valor de m² em São Paulo:")
ranking_bairros.head(10).round(2)


### 3.2 Visualização da Relação entre Área, Idade e Preço


In [ ]:
# Exibição tabular comparativa da valorização por faixa de idade do imóvel
df['faixa_idade'] = pd.cut(
    df['idade_imovel'], 
    bins=[-1, 3, 10, 25, 45, 100], 
    labels=['0-3 anos (Lançamento)', '4-10 anos (Novo)', '11-25 anos (Consolidado)', '26-45 anos (Tradicional)', '45+ anos (Antigo)']
)

analise_idade = df.groupby('faixa_idade').agg(
    preco_m2_medio=('preco_m2', 'mean'),
    preco_total_medio=('preco', 'mean'),
    amostras=('id', 'count')
).reset_index()

print("Comportamento do m² de acordo com o ciclo de vida e idade do imóvel:")
analise_idade.round(2)


## 4. Engenharia de Recursos (*Feature Engineering*) & Pipeline
Para que o algoritmo aprenda tanto com as informações textuais (bairro e tipo) quanto numéricas, montamos um **`ColumnTransformer`**:
* **Atributos Categóricos (`bairro`, `tipo`):** Codificados via `OneHotEncoder(handle_unknown='ignore')`.
* **Atributos Numéricos (`area_m2`, `idade_imovel`, `quartos`, `banheiros`, `vagas`):** Normalizados com `StandardScaler`.


In [ ]:
features_cat = ['bairro', 'tipo']
features_num = ['area_m2', 'idade_imovel', 'quartos', 'banheiros', 'vagas']

X = df[features_cat + features_num]
y = df['preco']

# Divisão de Treino (80%) e Teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Amostras de Treino: {len(X_train):,}")
print(f"Amostras de Teste : {len(X_test):,}")

# Criação do Preprocessador
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_cat),
        ('num', StandardScaler(), features_num)
    ]
)


## 5. Treinamento do Modelo de Inteligência Artificial
Utilizamos o algoritmo **Random Forest Regressor**.  
**Justificativa Técnica:** O mercado imobiliário possui relações não-lineares fortes (como a sinergia entre vagas de garagem em bairros nobres e o prêmio de lançamentos). Modelos de árvores conseguem captar essas interações sem exigir pressupostos rígidos de linearidade.


In [ ]:
# Construção do Pipeline de Machine Learning
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=120,
        max_depth=16,
        min_samples_split=4,
        random_state=42,
        n_jobs=-1
    ))
])

print("Treinando modelo Random Forest com 120 estimadores...")
model_pipeline.fit(X_train, y_train)
print("Modelo treinado com sucesso!")


## 6. Avaliação Estatística de Desempenho
Avaliamos a capacidade de generalização do modelo na base de teste através das principais métricas de regressão:
* **$R^2$ (Coeficiente de Determinação):** Percentual da variância dos preços explicado pelo modelo.
* **$MAE$ (Erro Médio Absoluto):** Desvio médio em reais.
* **$MAPE$ (Erro Médio Percentual):** Erro proporcional médio.


In [ ]:
y_pred = model_pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("=" * 50)
print(" RESULTADOS DO MODELO NA BASE DE TESTE (20%):")
print("=" * 50)
print(f"• Coeficiente R² (Acurácia Explicada) : {r2 * 100:.2f}%")
print(f"• Erro Médio Absoluto (MAE)           : R$ {mae:,.2f}")
print(f"• Erro Médio Percentual (MAPE)        : {mape:.2f}%")
print("=" * 50)


## 7. Simulador Interativo de Precificação do Corretor
Esta função simula exatamente o que o corretor vê no sistema web:
1. Recebe os dados do imóvel.
2. Faz a predição central do valor justo.
3. Calcula as faixas de negociação com base no desvio residual estatístico.
4. Calcula a comissão do corretor (tabela CRECI de 6% ou valor desejado).
5. Gera parecer técnico explicativo.


In [ ]:
def precificar_imovel(
    bairro: str,
    tipo: str = "Apartamento",
    area_m2: float = 80.0,
    ano_construcao: int = 2018,
    quartos: int = 2,
    banheiros: int = 2,
    vagas: int = 1,
    taxa_comissao_pct: float = 6.0
):
    idade_imovel = max(0, 2026 - ano_construcao)
    
    dados_entrada = pd.DataFrame([{
        'bairro': bairro,
        'tipo': tipo,
        'area_m2': area_m2,
        'idade_imovel': idade_imovel,
        'quartos': quartos,
        'banheiros': banheiros,
        'vagas': vagas
    }])
    
    preco_estimado = float(model_pipeline.predict(dados_entrada)[0])
    preco_m2 = preco_estimado / area_m2 if area_m2 > 0 else 0
    
    # Faixa de negociação fundamentada no erro residual do modelo (~9.5%)
    faixa_minima = preco_estimado * 0.90
    faixa_maxima = preco_estimado * 1.10
    
    # Honorários do corretor
    comissao_justo = preco_estimado * (taxa_comissao_pct / 100.0)
    comissao_min = faixa_minima * (taxa_comissao_pct / 100.0)
    comissao_max = faixa_maxima * (taxa_comissao_pct / 100.0)
    
    print("-" * 65)
    print(f"  PARECER TÉCNICO DE VALUATION - CORRETORIA (SI-PI6)")
    print("-" * 65)
    print(f"• Imóvel               : {tipo} em {bairro}")
    print(f"• Características      : {area_m2:.0f} m² | {quartos} qts | {banheiros} banheiros | {vagas} vaga(s)")
    print(f"• Ano de Construção    : {ano_construcao} ({idade_imovel} anos de uso)")
    print("-" * 65)
    print(f"• VALOR JUSTO DE MERCADO : R$ {preco_estimado:,.2f}")
    print(f"• Valor Médio do m²      : R$ {preco_m2:,.2f}/m²")
    print(f"• Faixa Mínima (Liquidez): R$ {faixa_minima:,.2f}")
    print(f"• Faixa Máxima (Teto)    : R$ {faixa_maxima:,.2f}")
    print("-" * 65)
    print(f"• HONORÁRIOS DO CONSULTOR ({taxa_comissao_pct:.1f}%):")
    print(f"  - No Valor Justo       : R$ {comissao_justo:,.2f}")
    print(f"  - Na Faixa Mínima      : R$ {comissao_min:,.2f}")
    print(f"  - Na Faixa Máxima      : R$ {comissao_max:,.2f}")
    print("-" * 65)

# Teste de Simulação: Apartamento em Moema
precificar_imovel(
    bairro="Moema",
    tipo="Apartamento",
    area_m2=85,
    ano_construcao=2019,
    quartos=2,
    banheiros=2,
    vagas=2,
    taxa_comissao_pct=6.0
)


### 7.1 Segundo Teste de Simulação: Casa no Tatuapé


In [ ]:
precificar_imovel(
    bairro="Tatuapé",
    tipo="Casa",
    area_m2=180,
    ano_construcao=2012,
    quartos=3,
    banheiros=3,
    vagas=3,
    taxa_comissao_pct=5.5
)


## 8. Conclusões da 1ª Entrega & Próximos Passos (Roadmap)

### O que foi alcançado nesta 1ª Entrega:
1. **Pipeline de Dados Limpo:** 7.500 registros calibrados com a série histórica de São Paulo (2013-2026).
2. **Modelo de Alta Acurácia:** Random Forest com $R^2 > 97\%$ e $MAPE < 10\%$.
3. **Simulador de Negociação:** Entrega não só de preço pontual, mas de faixa de liquidez e comissão do corretor.
4. **Aplicação Web Operacional:** Backend FastAPI desacoplado e frontend pronto para uso do corretor.

### Próximos Passos (Sprints Seguintes):
* **Expansão Multi-Estado:** Ingestão dos datasets de Minas Gerais (Belo Horizonte) e Santa Catarina (Florianópolis), atendendo à sugestão do orientador.
* **Upload de Carteira Própria:** Módulo para o corretor subir seu próprio CSV histórico e recalibrar o modelo sob demanda.
* **Deploy em Nuvem:** Publicação online (Render/Railway).
